## Import libraries

In [ ]:

import pandas as pd
import numpy as np
import re
from datetime import datetime
from google_play_scraper import app, reviews, Sort
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))
from scripts.text_cleaner import clean_text
from scripts.preprocessing import normalize_dates, remove_duplicates, validate_ratings

## Web Scraping

In [3]:
CBE_APP_ID = 'com.combanketh.mobilebanking'

# app metadata 
app_info = app(
    CBE_APP_ID,
    lang='en', 
    country='et'  
)

print("=" * 50)
print("CBE Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

CBE Bank App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.286519
Total Ratings: 48,255
Total Reviews: 9,308
Installs     : 5,000,000+


## Collecting reviews

In [4]:
print(f"Scraping reviews for CBE...")

result, continuation_token = reviews(
    CBE_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       
    count=500,              
    filter_score_with=None 
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for CBE...
Collected 500 raw reviews


## Inspecting data

In [46]:
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: dd4ae5b5-f5a4-42e9-a526-5fe9387dd7a4
  userName: Amha
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocK__JypXyqiLGNdWqfOq4FJDJKjZFy3Oyjx6N06TbEjVRFIMw=mo
  content: good
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.2.4
  at: 2026-05-12 19:02:00
  replyContent: None
  repliedAt: None
  appVersion: 5.2.4


## Extracting needed fields

In [47]:
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Commercial Bank of Ethiopia',
        'source'   : 'Google Play'
    })

df = pd.DataFrame(raw_data)

print(f"Shape: {df.shape}")
df.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,dd4ae5b5-f5a4-42e9-a526-5fe9387dd7a4,good,5,2026-05-12 19:02:00,Commercial Bank of Ethiopia,Google Play
1,e9233eb9-c338-4c80-8f21-26cd31acd65f,Good to use,5,2026-05-12 12:23:58,Commercial Bank of Ethiopia,Google Play
2,6b0d612d-f9c1-4cac-bb37-b945c97a2e9a,cbe,1,2026-05-12 06:18:40,Commercial Bank of Ethiopia,Google Play
3,68ff6a46-3659-47f0-a20f-fba5755a3f67,Cbe,4,2026-05-11 17:21:59,Commercial Bank of Ethiopia,Google Play
4,95f3c528-c059-4a5a-8973-46182fe5cb22,best and secured,5,2026-05-11 11:04:31,Commercial Bank of Ethiopia,Google Play


## Exploring the Raw Data

In [48]:
print(f"Total reviews collected: {len(df)}")
print(f"\nColumn dtypes:")
print(df.dtypes)

Total reviews collected: 500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object


## Rating distribution

In [50]:
print("Rating distribution:")
rating_counts = df['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  339  ███████████████████████████████████████████████████████████████████
  4 stars:   41  ████████
  3 stars:   35  ███████
  2 stars:   11  ██
  1 stars:   74  ██████████████


## Checking date column

In [51]:
print("Sample date values (raw):")
print(df['date'].head(10).to_string())

print(f"\nDate dtype: {df['date'].dtype}")

Sample date values (raw):
0   2026-05-12 19:02:00
1   2026-05-12 12:23:58
2   2026-05-12 06:18:40
3   2026-05-11 17:21:59
4   2026-05-11 11:04:31
5   2026-05-11 09:22:25
6   2026-05-11 00:24:09
7   2026-05-10 17:46:00
8   2026-05-10 16:13:18
9   2026-05-10 15:07:21

Date dtype: datetime64[ns]


## Missing Values

In [52]:
print("Missing Values")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

for col in df.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "Nothing missing :)"
    print(f"  {col:<15}: {status}")

Missing Values
  review_id      : Nothing missing :)
  review         : Nothing missing :)
  rating         : Nothing missing :)
  date           : Nothing missing :)
  bank           : Nothing missing :)
  source         : Nothing missing :)


## Duplicate Reviews

In [53]:
print("Duplicates")

exact_dupes = df.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

id_dupes = df.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

empty_reviews = (df['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Duplicates
  Exact duplicate reviews : 127
  Duplicate review IDs    : 0
  Empty review texts      : 0


## Remove Duplicates

In [54]:
df = remove_duplicates(df)

Removed 0 duplicate reviews


## Date Format

In [55]:
print("Date Format")
print(f"  Current dtype: {df['date'].dtype}")
print(f"  Sample values: {df['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Date Format
  Current dtype: datetime64[ns]
  Sample values: 2026-05-12 19:02:00
  Target format: YYYY-MM-DD (string or date object)


## Normalize dates

In [56]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

df = normalize_dates(df)

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-12 19:02:00
1   2026-05-12 12:23:58
2   2026-05-12 06:18:40
dtype: datetime64[ns]

After normalization:
0    2026-05-12
1    2026-05-12
2    2026-05-12
dtype: object

Date range: 2026-03-01 to 2026-05-12


## Clean Review Text

In [57]:
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


## Rating Validation

In [58]:
df = validate_ratings(df)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Removed 0 invalid ratings
Remaining: 500 reviews
Rating dtype: int64


## Cleaned data

In [59]:
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,good,5,2026-05-12,Commercial Bank of Ethiopia,Google Play
1,cbe,1,2026-05-12,Commercial Bank of Ethiopia,Google Play
2,Good to use,5,2026-05-12,Commercial Bank of Ethiopia,Google Play
3,Cbe,4,2026-05-11,Commercial Bank of Ethiopia,Google Play
4,best and secured,5,2026-05-11,Commercial Bank of Ethiopia,Google Play
5,best,5,2026-05-11,Commercial Bank of Ethiopia,Google Play
6,ok,5,2026-05-11,Commercial Bank of Ethiopia,Google Play
7,good,5,2026-05-10,Commercial Bank of Ethiopia,Google Play
8,why is my transaction fee hidden? Why doesn't ...,3,2026-05-10,Commercial Bank of Ethiopia,Google Play
9,Gal✅😶‍🌫️,1,2026-05-10,Commercial Bank of Ethiopia,Google Play


## Save cleaned data

In [60]:
output_path = '../data/processed/cbe_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/cbe_reviews_clean.csv
